# Tutorial: Layer 4 KBO Translation Model


## 1. 목적

            4번 레이어는 **KBO 번역 모델**이다.

            핵심 질문은 이것이다.

            > MLB/AAA/AA/NPB에서 보인 능력이 KBO에서 얼마나 그대로 먹히는가?

            ## 사용한 모델/방법

            - role prior baseline: 포지션/역할별 과거 평균 성공률
            - ridge logistic regression: 작은 표본에서 과적합을 줄이는 설명 가능한 분류 모델
            - balanced ridge logistic: 성공/실패 불균형을 보정한 로지스틱 모델
            - shallow random forest: 비선형 상호작용을 약하게 허용한 모델
            - repeated stratified CV: 작은 표본에서 fold 운을 줄이기 위해 반복 교차검증
            - Brier/logloss/AUC/top-k precision: 확률 예측과 순위 성능을 함께 확인


## 2. 가장 중요한 결론

            이 레이어의 결론은 "복잡한 모델이 최고다"가 아니다.

            오히려 현재 데이터에서는 복잡한 classifier를 직접 후보 점수로 승격하지 않는다는 판단이 핵심이다.

            > KBO 번역 모델은 후보 점수 엔진이 아니라, 성공/실패 가능성을 조심스럽게 보정하는 진단 장치다.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate[gate["gate"].eq("G4")], ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"])


In [ ]:
audit = read_table("kbo_translation_retrain_gate_audit_v0_3.csv")
take_cols(audit, ["gate", "layer", "check", "pass_rows", "total_rows", "status", "blocking_gap"], n=10)


## 3. 반복 교차검증 결과 읽기

            promotion_status가 핵심이다.

            - baseline: 비교 기준
            - do_not_promote: 연구 참고로는 쓰지만 후보 점수로 직접 쓰지 않음
            - promote가 없다는 것은 모델링 실패가 아니라 과대해석 방지다.


In [ ]:
cv = read_table("kbo_translation_failure_repeated_cv_comparison_v0_3.csv")
summary = count_by(cv, ["role_model_family", "target", "promotion_status"])
summary


In [ ]:
take_cols(
    cv.sort_values(["role_model_family", "target", "model"]),
    ["role_model_family", "target", "model", "folds", "total_valid_rows", "mean_auc", "mean_brier", "mean_precision_top_25pct", "brier_lift_vs_role_prior", "top25_precision_lift_vs_role_prior", "promotion_status"],
    n=18,
)


## 4. v0.2와 v0.3 비교

            새 데이터를 추가했을 때 모델이 좋아지는지, 아니면 표본은 늘었지만 안정성은 떨어지는지 확인한다.


In [ ]:
compare = read_table("kbo_translation_failure_v0_2_vs_v0_3_comparison.csv")
take_cols(
    compare,
    ["role_model_family", "target", "model", "total_valid_rows_delta_v0_3_minus_v0_2", "mean_auc_delta_v0_3_minus_v0_2", "mean_brier_delta_v0_3_minus_v0_2", "brier_direction", "promotion_status_v0_3"],
    n=18,
)


## 5. 발표용 한 줄

            4번 레이어는 해외 기록을 KBO 성과로 번역하려 했지만, 작은 표본에서 복잡한 모델이 안정적으로 승격되지 않았기 때문에 보수적으로 role-prior와 feature-contract 진단을 사용했다.

            ## 연습문제

            mean_auc가 높아도 promotion_status가 do_not_promote인 이유를 Brier/logloss와 표본 크기 관점에서 설명해보자.
